# 2 · Entrenar el LoRA de Lía (SDXL)
Antes: mueve a `Mi unidad/lia/dataset` las **20–30 imágenes** donde se vea la misma cara (desde `lia/candidatas`).
**GPU T4.** Tiempo aproximado: 1 h (estimado, no medido). Si Colab corta la sesión, vuelve a ejecutar:
el resultado final queda en `lia/lora`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
BASE = '/content/drive/MyDrive/lia'      # todo se guarda aquí, en tu Google Drive
for d in ('candidatas', 'dataset', 'lora', 'salida'):
    os.makedirs(f'{BASE}/{d}', exist_ok=True)
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NINGUNA -> Entorno de ejecución > Cambiar tipo > GPU T4')

In [ ]:
!pip install -q diffusers==0.31.0 transformers accelerate peft safetensors

In [ ]:
!pip install -q bitsandbytes
!wget -q -O train_dreambooth_lora_sdxl.py https://raw.githubusercontent.com/huggingface/diffusers/v0.31.0/examples/dreambooth/train_dreambooth_lora_sdxl.py
import glob
n = len(glob.glob(f'{BASE}/dataset/*.png') + glob.glob(f'{BASE}/dataset/*.jpg'))
print(n, 'imágenes en el dataset')
assert n >= 10, 'Mueve primero tus mejores imágenes a lia/dataset (mínimo 10, ideal 20-30)'

In [ ]:
!accelerate launch --mixed_precision=fp16 train_dreambooth_lora_sdxl.py \
  --pretrained_model_name_or_path=SG161222/RealVisXL_V4.0 \
  --pretrained_vae_model_name_or_path=madebyollin/sdxl-vae-fp16-fix \
  --instance_data_dir={BASE}/dataset \
  --instance_prompt="photo of lia_character woman" \
  --output_dir={BASE}/lora \
  --resolution=768 --center_crop \
  --train_batch_size=1 --gradient_accumulation_steps=2 \
  --gradient_checkpointing --use_8bit_adam --mixed_precision=fp16 \
  --learning_rate=1e-4 --lr_scheduler=constant --lr_warmup_steps=0 \
  --rank=16 --max_train_steps=1000 --checkpointing_steps=500 --seed=42

In [ ]:
import os
f = f'{BASE}/lora/pytorch_lora_weights.safetensors'
print('OK, LoRA guardado:' if os.path.exists(f) else 'NO se creó el LoRA, revisa los errores de arriba:', f)